In [10]:

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
from gymnasium import spaces
from collections import deque

In [16]:
TRAINING_ITERATIONS = 400

In [ ]:
# 1. Custom Gymnasium Environment
class ComparisonEnv(gym.Env):
    def __init__(self):
        super().__init__()
        # State space: 2 dimension vector with values from -1 to 1
        self.observation_space = spaces.Box(low=-1, high=1, shape=(24,), dtype=np.float32)
        # Action space: 0, 1 representing whether the first number is higher or not
        self.action_space = spaces.Discrete(2)
        self.state = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = np.random.uniform(-1.0, 1.0, size=(2,)).astype(np.float32)
        return self.state, {}

    def step(self, action):
        x1, x2 = self.state
        # Optimal choice: action 1 if x1 > x2, else action 0
        optimal_action = 1 if x1 > x2 else 0
        reward = 1.0 if action == optimal_action else -1.0
        
        # Single-step episode
        terminated = True
        truncated = False
        return self.state, reward, terminated, truncated, {}

In [12]:
# 2. Q-Network Architecture
class DQN(nn.Module):
    def __init__(self, state_dim=2, action_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, action_dim)
        )

    def forward(self, x):
        return self.net(x)

In [13]:
# 3. Environment & Model Initialization
env = ComparisonEnv()
q_net = DQN()
target_net = DQN()
target_net.load_state_dict(q_net.state_dict())

optimizer = optim.Adam(q_net.parameters(), lr=1e-3)
memory = deque(maxlen=5000)

gamma = 0.99
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.99
batch_size = 32

In [ ]:
# 4. Training Loop
for episode in range(TRAINING_ITERATIONS):
    state, _ = env.reset()
    done = False
    
    while not done:
        # Epsilon-greedy action selection
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0)
                action = q_net(state_t).argmax(dim=1).item()

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        memory.append((state, action, reward, next_state, done))

        # Replay Buffer Sampling & Network Update
        if len(memory) >= batch_size:
            batch = random.sample(memory, batch_size)
            s_arr, a_arr, r_arr, ns_arr, d_arr = zip(*batch)

            states = torch.FloatTensor(np.array(s_arr))
            actions = torch.LongTensor(a_arr).unsqueeze(1)
            rewards = torch.FloatTensor(r_arr)
            next_states = torch.FloatTensor(np.array(ns_arr))
            dones = torch.FloatTensor(d_arr)

            # Compute current Q(s, a)
            q_values = q_net(states).gather(1, actions).squeeze(1)

            # Target Q-value using Bellman Equation
            with torch.no_grad():
                max_next_q = target_net(next_states).max(1)[0]
                targets = rewards + gamma * max_next_q * (1 - dones)

            loss = nn.MSELoss()(q_values, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    if episode % 20 == 0:
        target_net.load_state_dict(q_net.state_dict())

In [15]:
# 5. Verification Test
def test_input(x1, x2):
    q_net.eval()
    with torch.no_grad():
        inp = torch.FloatTensor([x1, x2]).unsqueeze(0)
        q_vals = q_net(inp).numpy()[0]
        chosen_action = int(q_vals.argmax())
    return chosen_action, q_vals

print("Test [1.0, 0.0]  -> Selected Action:", test_input(1, 0.0)[0])  # Expected: 1
print("Test [-0.5, 0.2] -> Selected Action:", test_input(-0.5, 0.2)[0]) # Expected: 0

Test [1.0, 0.0]  -> Selected Action: 1
Test [-0.5, 0.2] -> Selected Action: 0
